<a href="https://colab.research.google.com/github/AstronomekDomansky/-.-1/blob/main/%D0%A1%D0%B8%D0%B8%D0%91%D0%945.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# ⬇️  ЗАПОЛНИТЕ ЭТИ ТРИ ПОЛЯ СВОИМИ ДАННЫМИ
# ==========================================================
last_name   = "Арутюнян"    # <-- впишите свою фамилию
first_name  = "Дмитрий"      # <-- впишите своё имя
list_number = 2           # <-- ваш номер по списку (целое число)
# ==========================================================

import hashlib

def make_seed(last: str, first: str, number: int) -> int:
    # ФИО + номер -> стабильный seed 0..99999
    key = f"{last.strip().lower()}|{first.strip().lower()}|{int(number)}"
    return int(hashlib.sha256(key.encode("utf-8")).hexdigest(), 16) % 100_000

SEED = make_seed(last_name, first_name, list_number)

print("Студент:", last_name, first_name, f"(№{list_number})")
print("Ваш персональный SEED:", SEED)
print("Запишите SEED в итоговый отчёт — он подтверждает вариант.")


Студент: Арутюнян Дмитрий (№2)
Ваш персональный SEED: 36390
Запишите SEED в итоговый отчёт — он подтверждает вариант.


> **Как это работает.** `hashlib.sha256` переводит строку с ФИО и номером в большое число, затем берётся остаток от деления на 100 000. Одинаковые ФИО и номер всегда дают один и тот же SEED, разные — почти всегда разный. Весь датасет ниже зависит только от этого числа.


## Подготовка окружения

В Google Colab пакеты уже есть. Локально при необходимости: `pip install pandas numpy matplotlib seaborn`.


In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)


## Генерация вашего набора данных

Ячейка ниже создаёт **ваш** журнал обращений IT-службы. Разбирать генератор не нужно — достаточно запустить.

Внутри специально заложены пропуски, выбросы, дубликаты и разный регистр текста, иначе предобработка была бы формальной.

**Признаки**

| Столбец | Смысл |
|---------|-------|
| `ticket_id` | идентификатор обращения |
| `opened_ts` | время открытия в Unix-секундах |
| `department` | отдел: Support, Backend, DevOps, Security, Data |
| `product` | продукт: Cloud, CRM, Billing, Auth, Analytics |
| `priority` | приоритет: Low, Medium, High, Critical |
| `channel` | канал: Email, Slack, Portal, Phone |
| `assignee_level` | уровень исполнителя: Junior, Middle, Senior |
| `client_tier` | тариф клиента: Free, Pro, Enterprise |
| `cpu_pct` | загрузка CPU на затронутом узле, % |
| `memory_mb` | занятая память, МБ |
| `n_reopens` | сколько раз тикет открывали повторно |
| `comments` | число комментариев в тикете |
| `resolve_hours` | часы до закрытия |
| **`sla_breached`** | **цель**: 1 — SLA нарушен, 0 — нет |


In [ ]:
#@title Создание персонального датасета (нажмите ▶️) { display-mode: "form" }

def load_it_tickets(seed: int, n: int = 1800) -> "pd.DataFrame":
    """Индивидуальный «грязный» журнал IT-обращений по seed."""
    rng = np.random.default_rng(seed)

    opened_ts = rng.integers(1_735_689_600, 1_751_241_600, size=n)
    department = rng.choice(
        ["Support", "Backend", "DevOps", "Security", "Data"],
        size=n, p=[0.32, 0.22, 0.18, 0.14, 0.14],
    )
    product = rng.choice(
        ["Cloud", "CRM", "Billing", "Auth", "Analytics"], size=n
    )
    priority = rng.choice(
        ["Low", "Medium", "High", "Critical"], size=n, p=[0.40, 0.30, 0.20, 0.10]
    )
    channel = rng.choice(["Email", "Slack", "Portal", "Phone"], size=n)
    assignee = rng.choice(["Junior", "Middle", "Senior"], size=n, p=[0.35, 0.45, 0.20])
    tier = rng.choice(["Free", "Pro", "Enterprise"], size=n, p=[0.45, 0.40, 0.15])
    cpu_pct = np.clip(np.round(rng.normal(48, 18, size=n), 1), 1, 99)
    memory_mb = np.clip(np.round(rng.normal(2200, 850, size=n), 0), 256, None)
    n_reopens = rng.poisson(0.7, size=n)
    comments = rng.poisson(4.0, size=n) + 1
    resolve_hours = np.clip(np.round(rng.lognormal(1.55, 0.75, size=n), 2), 0.2, 240)

    logit = (
        -1.4
        + 1.4 * (priority == "Critical")
        + 0.7 * (priority == "High")
        - 0.4 * (priority == "Low")
        + 0.035 * resolve_hours
        + 0.018 * cpu_pct
        + 0.55 * (department == "Security")
        + 0.35 * (n_reopens >= 2)
        - 0.45 * (assignee == "Senior")
        - 0.30 * (tier == "Enterprise")
    )
    sla = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)

    df = pd.DataFrame({
        "ticket_id": np.arange(10_000, 10_000 + n),
        "opened_ts": opened_ts,
        "department": department,
        "product": product,
        "priority": priority,
        "channel": channel,
        "assignee_level": assignee,
        "client_tier": tier,
        "cpu_pct": cpu_pct,
        "memory_mb": memory_mb,
        "n_reopens": n_reopens,
        "comments": comments,
        "resolve_hours": resolve_hours,
        "sla_breached": sla,
    })

    df.loc[rng.choice(n, int(0.06 * n), replace=False), "resolve_hours"] = np.nan
    df.loc[rng.choice(n, int(0.04 * n), replace=False), "cpu_pct"] = np.nan
    df.loc[rng.choice(n, int(0.03 * n), replace=False), "comments"] = np.nan
    df.loc[rng.choice(n, 10, replace=False), "resolve_hours"] *= 9
    df.loc[rng.choice(n, 8, replace=False), "memory_mb"] *= 7
    ridx = rng.choice(n, int(0.12 * n), replace=False)
    df.loc[ridx, "department"] = df.loc[ridx, "department"].str.upper()
    pidx = rng.choice(n, int(0.08 * n), replace=False)
    df.loc[pidx, "product"] = "  " + df.loc[pidx, "product"].str.lower() + " "
    df = pd.concat([df, df.sample(int(0.025 * n), random_state=seed)], ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

print("Функция load_it_tickets готова.")


Функция load_it_tickets готова.


Загрузите **ваш** набор одной строкой:


In [ ]:
df_raw = load_it_tickets(SEED)

print("Ваш датасет создан. Размер:", df_raw.shape)
print("Доля нарушений SLA:", f"{df_raw['sla_breached'].mean():.1%}")


Ваш датасет создан. Размер: (1845, 14)
Доля нарушений SLA: 44.2%


> В таблице уже спрятаны пропуски, выбросы, дубликаты и неаккуратный текст. Найти и поправить их — часть заданий. Дальше везде работайте со **своим** `df_raw`, если в блоке не сказано иное.


---
# Этап 5. Подготовка данных и признаки (практика 5)

**Цель.** Убрать дубликаты, закрыть пропуски, привести текст к одному виду, закодировать категории, масштабировать числа, собрать несколько новых признаков. Это тот же конвейер, что на практике 5, только на вашем журнале тикетов.

Работайте с копией: исходный `df_raw` не затирайте.


### Пример


In [ ]:
df = df_raw.copy()
before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print("Удалено дубликатов:", before - df.shape[0])
print("Пропуски по столбцам:")
print(df.isna().sum())

median_cpu = df["cpu_pct"].median()
df["cpu_pct"] = df["cpu_pct"].fillna(median_cpu)
print("Пропусков в cpu_pct после заполнения:", int(df["cpu_pct"].isna().sum()))


In [ ]:
# Текст: пробелы и регистр. Без этого Backend и BACKEND — разные категории.
demo = df["department"].astype(str).str.strip().str.title()
print("Уникальные отделы до:", df["department"].nunique())
print("После strip+title:", demo.nunique())
print(sorted(demo.unique()))


### Задание 5

Начните заново с копии `df_raw`. Сохраните результат в `df_clean`, матрицу признаков в `X`, цель в `y`.

1. Удалите дубликаты.
2. Заполните пропуски: `resolve_hours` и `cpu_pct` — **медианой**, `comments` — **медианой** или нулём. Проверьте, что пропусков не осталось.
3. Приведите `department` и `product` к одному виду: `.str.strip()` и единый регистр (например `.str.title()`). Снова посчитайте число уникальных значений.
4. Закодируйте категории `department`, `product`, `priority`, `channel`, `assignee_level`, `client_tier` через `pd.get_dummies(...)`.
5. Создайте **не меньше трёх** осмысленных признаков, например:
   - `is_critical` — флаг приоритета Critical;
   - `load_score` — `cpu_pct * memory_mb / 1000`;
   - `reopen_rate` — `n_reopens / comments` (защититесь от деления на ноль);
   - свой признак (выходной ли день из `opened_ts`, «долгий тикет» и т. п.).
6. Отберите числовые столбцы (без `ticket_id` и `opened_ts`) и масштабируйте их `StandardScaler`. Сложите с dummy-признаками в `df_clean`.
7. Соберите `X` (всё, кроме `sla_breached` и идентификаторов) и `y = df_clean["sla_breached"]`. Напечатайте формы.


In [ ]:
# ▼▼▼ ВАШ КОД (Задание 5) ▼▼▼
from sklearn.preprocessing import StandardScaler

df = df_raw.copy()

# 1–2. Дубликаты и пропуски


In [ ]:
# 3. Текст department и product


In [ ]:
# 4. get_dummies


In [ ]:
# 5. Новые признаки (минимум 3, один — свой)


In [ ]:
# 6–7. Масштаб, df_clean, X, y
df_clean = None
X = None
y = None

# print("Размер X:", X.shape, "| Размер y:", y.shape)

# ▲▲▲ КОНЕЦ ВАШЕГО КОДА ▲▲▲


**Контрольные вопросы к этапу 5:**

1. Когда для пропусков берут медиану, а когда среднее?
2. Что такое One-Hot (`get_dummies`) и почему отделы нельзя просто заменить числами 1, 2, 3…?
3. Зачем масштабировать признаки перед линейными моделями и kNN и почему дереву это обычно не нужно?
4. Почему `ticket_id` нельзя подавать в модель как обычный признак?


*Ответы:*

1.

2.

3.

4.
